In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ABSORA AI HUB — T4x2 DUAL GPU 3-MODEL PARALLEL ORCHESTRATOR
# ═══════════════════════════════════════════════════════════════════════════════
#
#  GPU Layout (32 GB total VRAM):
#  ┌─────────────────────────────────────────────────────────┐
#  │ GPU 0 (16 GB T4)                                        │
#  │   Slot 0 → DeepSeek-R1-Distill-Qwen-1.5B  ~3 GB FP16  │
#  │   Slot 1 → Phi-3.5-mini-instruct           ~8 GB FP16  │
#  │   Combined ≈ 14.5 GB  ✓                               │
#  ├─────────────────────────────────────────────────────────┤
#  │ GPU 1 (16 GB T4)                                        │
#  │   Slot 2 → Qwen2.5-7B-Instruct-AWQ        ~4.2 GB AWQ │
#  │   KV cache ≈ 10 GB  ✓                                  │
#  └─────────────────────────────────────────────────────────┘
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, time, json, re, subprocess, threading

# ── Tesla T4 (sm_75) CUDA Compatibility Flags ──────────────────────────
os.environ['VLLM_ATTENTION_BACKEND']      = 'TORCH_SDPA'
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
os.environ['TORCH_CUDA_ARCH_LIST']        = '7.5'

# ── Zero-Config Defaults ──────────────────────────────────────────────────────
WEBHOOK_URL      = os.environ.get('WEBHOOK_URL',  'https://absora-ai-hub.vercel.app/api/webhook')
SESSION_ID       = os.environ.get('SESSION_ID',   'kaggle-t4x2-session')
ABSORA_SECRET    = os.environ.get('ABSORA_SECRET', 'absora-secret-key-2026')

# ── Static Model Registry ──────────────────────────────────────────────────────
MODEL_REGISTRY = {
    'deepseek-r1-1.5b': {
        'hf_id':        'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
        'gpu_slot':      0,
        'gpu_util':     '0.22',
        'max_len':       '8192',
        'dtype':         'float16',
        'port':          8001,
    },
    'phi3.5-mini': {
        'hf_id':        'microsoft/Phi-3.5-mini-instruct',
        'gpu_slot':      0,
        'gpu_util':     '0.72',
        'max_len':       '8192',
        'dtype':         'float16',
        'port':          8002,
    },
    'qwen2.5-7b': {
        'hf_id':        'Qwen/Qwen2.5-7B-Instruct-AWQ',
        'gpu_slot':      1,
        'gpu_util':     '0.88',
        'max_len':       '8192',
        'dtype':         'float16',
        'quantization': 'awq',
        'port':          8003,
    },
}

print(f'[Absora Worker] Starting 3-model T4x2 parallel session: {SESSION_ID}')
print(f'[Absora Worker] Webhook: {WEBHOOK_URL}')
print('[Absora Worker] GPU Layout: GPU0→DeepSeek+Phi | GPU1→Qwen2.5-7B AWQ')

# ── Install T4-Compatible vLLM (0.6.4.post1) Dependencies ───────────────────
print('[Absora Worker] Installing vllm==0.6.4.post1, FastAPI, Uvicorn, autoawq...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'vllm==0.6.4.post1', 'fastapi', 'uvicorn[standard]',
    'autoawq', 'psutil', 'requests', 'httpx'
], check=True)

# ── Download Cloudflared ───────────────────────────────────────────────────────
if not os.path.exists('./cloudflared'):
    print('[Absora Worker] Downloading cloudflared...')
    subprocess.run([
        'curl', '-fsSL',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-o', 'cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

import requests, uvicorn, httpx, psutil
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse


In [ ]:
# ── 3-Model Parallel Orchestrator ─────────────────────────────────────────────

class TripleModelOrchestrator:
    def __init__(self):
        self.processes    = {}   # model_id → subprocess.Popen
        self.health       = {}   # model_id → bool
        self.last_traffic = time.time()
        self.lock         = threading.Lock()

    # ── Spawn a single vLLM process pinned to one GPU ──────────────────────────
    def _spawn(self, model_id: str) -> subprocess.Popen:
        cfg  = MODEL_REGISTRY[model_id]
        env  = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES']          = str(cfg['gpu_slot'])
        env['VLLM_ATTENTION_BACKEND']        = 'TORCH_SDPA'
        env['VLLM_USE_FLASHINFER_SAMPLER']   = '0'
        env['TORCH_CUDA_ARCH_LIST']        = '7.5'

        cmd = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model',                   cfg['hf_id'],
            '--served-model-name',       model_id,
            '--port',                    str(cfg['port']),
            '--tensor-parallel-size',    '1',
            '--gpu-memory-utilization',  cfg['gpu_util'],
            '--max-model-len',           cfg['max_len'],
            '--max-num-seqs',            '32',
            '--dtype',                   cfg['dtype'],
            '--enforce-eager',
        ]

        if 'quantization' in cfg:
            cmd.extend(['--quantization', cfg['quantization']])

        print(f'[Orchestrator] Spawning {model_id} ({cfg["hf_id"]}) on GPU {cfg["gpu_slot"]} → port {cfg["port"]}')
        return subprocess.Popen(cmd, env=env)

    # ── Wait for a model to report /health ────────────────────────────────────
    def _wait_healthy(self, model_id: str, proc: subprocess.Popen, timeout: int = 300) -> bool:
        port = MODEL_REGISTRY[model_id]['port']
        deadline = time.time() + timeout
        while time.time() < deadline:
            if proc.poll() is not None:
                print(f'[Orchestrator] ❌ {model_id} process exited early (code {proc.returncode})')
                return False
            try:
                r = requests.get(f'http://localhost:{port}/health', timeout=2)
                if r.status_code == 200:
                    print(f'[Orchestrator] ✅ {model_id} is HEALTHY on port {port}')
                    return True
            except:
                pass
            time.sleep(3)
        print(f'[Orchestrator] ❌ {model_id} failed to start within {timeout}s')
        return False

    # ── Load all 3 models sequentially (GPU-safe ordering) ────────────────────
    def start_all(self):
        print('[Orchestrator] ═══ Starting 3-model T4x2 boot sequence ═══')

        # Step 1: Spawn GPU 1 model (Qwen AWQ) immediately — independent GPU
        qwen_proc = self._spawn('qwen2.5-7b')
        self.processes['qwen2.5-7b'] = qwen_proc

        # Step 2: Spawn DeepSeek on GPU 0
        ds_proc = self._spawn('deepseek-r1-1.5b')
        self.processes['deepseek-r1-1.5b'] = ds_proc

        # Step 3: Wait for DeepSeek to be healthy BEFORE loading Phi on GPU 0
        ds_ok = self._wait_healthy('deepseek-r1-1.5b', ds_proc)
        self.health['deepseek-r1-1.5b'] = ds_ok

        # Step 4: Spawn Phi on GPU 0
        phi_proc = self._spawn('phi3.5-mini')
        self.processes['phi3.5-mini'] = phi_proc

        # Step 5: Wait for Phi and Qwen in parallel
        phi_ok  = self._wait_healthy('phi3.5-mini',  phi_proc)
        qwen_ok = self._wait_healthy('qwen2.5-7b',   qwen_proc)
        self.health['phi3.5-mini']  = phi_ok
        self.health['qwen2.5-7b']   = qwen_ok

        # Summary
        loaded = [m for m, ok in self.health.items() if ok]
        failed = [m for m, ok in self.health.items() if not ok]
        print(f'[Orchestrator] ✅ Active models: {loaded}')
        if failed:
            print(f'[Orchestrator] ⚠️  Failed models: {failed}')

    def get_port(self, model_id: str) -> int:
        return MODEL_REGISTRY.get(model_id, {}).get('port', 8001)

    def is_alive(self, model_id: str) -> bool:
        proc = self.processes.get(model_id)
        return proc is not None and proc.poll() is None

    def get_status(self) -> dict:
        models_status = []
        for model_id, cfg in MODEL_REGISTRY.items():
            alive = self.is_alive(model_id)
            models_status.append({
                'model_id':  model_id,
                'hf_id':     cfg['hf_id'],
                'port':      cfg['port'],
                'gpu':       cfg['gpu_slot'],
                'gpu_util':  cfg['gpu_util'],
                'alive':     alive,
                'healthy':   self.health.get(model_id, False),
            })
        return {
            'session_id':    SESSION_ID,
            'gpu_config':    'T4x2 (2 × 16 GB)',
            'total_vram_gb': 32,
            'models':        models_status,
        }

    def shutdown_all(self):
        for model_id, proc in list(self.processes.items()):
            try:
                proc.kill()
                proc.wait(timeout=5)
                print(f'[Orchestrator] Killed {model_id}')
            except:
                pass
        self.processes.clear()

orchestrator = TripleModelOrchestrator()


In [ ]:
# ── FastAPI Gateway ────────────────────────────────────────────────────────────

app          = FastAPI(title='Absora 3-Model T4x2 Gateway')
proxy_client = httpx.AsyncClient(timeout=180.0)

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.get('/status')
def status():
    return orchestrator.get_status()

@app.get('/v1/models')
def list_models():
    alive = [mid for mid in MODEL_REGISTRY if orchestrator.is_alive(mid)]
    return {
        'object': 'list',
        'data': [
            {'id': mid, 'object': 'model', 'owned_by': 'absora'}
            for mid in alive
        ]
    }

@app.api_route('/v1/{path:path}', methods=['GET', 'POST', 'PUT', 'DELETE'])
async def proxy(path: str, request: Request):
    orchestrator.last_traffic = time.time()
    body      = await request.body()
    json_body = {}
    if body:
        try:
            json_body = json.loads(body)
        except:
            pass

    requested = json_body.get('model', 'qwen2.5-7b')
    alias_map = {
        'deepseek': 'deepseek-r1-1.5b',
        'deepseek-r1': 'deepseek-r1-1.5b',
        'phi': 'phi3.5-mini',
        'phi-3.5': 'phi3.5-mini',
        'qwen': 'qwen2.5-7b',
        'qwen2.5': 'qwen2.5-7b',
    }
    target = alias_map.get(requested, requested)

    if not orchestrator.is_alive(target):
        alive = [m for m in MODEL_REGISTRY if orchestrator.is_alive(m)]
        if not alive:
            raise HTTPException(503, 'No models are currently active')
        target = alive[0]

    port = orchestrator.get_port(target)
    url  = f'http://localhost:{port}/v1/{path}'

    headers = dict(request.headers)
    headers.pop('host', None)

    proxy_req = proxy_client.build_request(
        method=request.method, url=url, headers=headers, content=body
    )
    resp = await proxy_client.send(proxy_req, stream=True)
    return StreamingResponse(resp.aiter_raw(), status_code=resp.status_code, headers=dict(resp.headers))


In [ ]:
# ── Cloudflare Quick Tunnel ────────────────────────────────────────────────────

tunnel_url = None

def start_tunnel():
    os.system('./cloudflared tunnel --url http://localhost:8000 > /tmp/tunnel.log 2>&1')

threading.Thread(target=start_tunnel, daemon=True).start()

print('[Absora Worker] Waiting for Cloudflare tunnel URL...')
for _ in range(30):
    time.sleep(2)
    if os.path.exists('/tmp/tunnel.log'):
        with open('/tmp/tunnel.log') as f:
            log = f.read()
        m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log)
        if m:
            tunnel_url = m.group(0)
            break

if tunnel_url:
    print(f'\n[Absora Worker] 🌐 TUNNEL: {tunnel_url}\n')
    try:
        requests.post(f'{WEBHOOK_URL}/tunnel', json={
            'session_id':   SESSION_ID,
            'tunnel_url':   tunnel_url,
            'gpu_config':   'T4x2',
            'models':       list(MODEL_REGISTRY.keys()),
        }, timeout=10)
    except Exception as e:
        print(f'[Absora Worker] Tunnel webhook failed: {e}')
else:
    print('[Absora Worker] ⚠️  Tunnel URL not detected — models still start normally')

# ── Start FastAPI (background) ─────────────────────────────────────────────────
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
).start()

# ── Boot All 3 Models ──────────────────────────────────────────────────────────
orchestrator.start_all()

# ── Post initial status ────────────────────────────────────────────────────────
try:
    requests.post(f'{WEBHOOK_URL}/status', json={
        **orchestrator.get_status(),
        'tunnel_url': tunnel_url,
    }, timeout=5)
except:
    pass

# ── Heartbeat + Auto-Shutdown (2-min idle) ─────────────────────────────────────
print('[Absora Worker] ✅ All systems active. 2-min idle auto-shutdown enabled.')
while True:
    time.sleep(15)
    idle_sec = time.time() - orchestrator.last_traffic

    heartbeat = {
        **orchestrator.get_status(),
        'tunnel_url':  tunnel_url,
        'idle_seconds': round(idle_sec),
    }

    try:
        requests.post(f'{WEBHOOK_URL}/status', json=heartbeat, timeout=5)
    except:
        pass

    if idle_sec > 120:
        print('[Absora Worker] 💤 2-minute idle timeout — shutting down all vLLM processes...')
        try:
            requests.post(f'{WEBHOOK_URL}/stopped', json=heartbeat, timeout=5)
        except:
            pass
        orchestrator.shutdown_all()
        sys.exit(0)
